In [1]:
import pandas as pd

df = pd.read_csv("deliveries.csv")

print(df.shape)
df.head()


(260920, 17)


,match_id,inning,batting_team,bowling_team,over,ball,batter,bowler,non_striker,batsman_runs,extra_runs,total_runs,extras_type,is_wicket,player_dismissed,dismissal_kind,fielder
0,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,1,SC Ganguly,P Kumar,BB McCullum,0,1,1,legbyes,0,NaN,NaN,NaN
1,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,2,BB McCullum,P Kumar,SC Ganguly,0,0,0,NaN,0,NaN,NaN,NaN
2,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,3,BB McCullum,P Kumar,SC Ganguly,0,1,1,wides,0,NaN,NaN,NaN
3,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,4,BB McCullum,P Kumar,SC Ganguly,0,0,0,NaN,0,NaN,NaN,NaN
4,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,5,BB McCullum,P Kumar,SC Ganguly,0,0,0,NaN,0,NaN,NaN,NaN


In [2]:
player_match = (
    df.groupby(["match_id", "batter"])
    .agg(
        runs=("batsman_runs", "sum"),
        balls_faced=("ball", "count"),
        got_out=("is_wicket", "max")
    )
    .reset_index()
)

player_match.head()


,match_id,batter,runs,balls_faced,got_out
0,335982,AA Noffke,9,12,1
1,335982,B Akhil,0,2,1
2,335982,BB McCullum,158,77,0
3,335982,CL White,6,10,1
4,335982,DJ Hussey,12,12,1


In [3]:
# Strike Rate = (runs / balls) * 100
player_match["strike_rate"] = (
    player_match["runs"] / player_match["balls_faced"]
) * 100

player_match.head()


,match_id,batter,runs,balls_faced,got_out,strike_rate
0,335982,AA Noffke,9,12,1,75.000000
1,335982,B Akhil,0,2,1,0.000000
2,335982,BB McCullum,158,77,0,205.194805
3,335982,CL White,6,10,1,60.000000
4,335982,DJ Hussey,12,12,1,100.000000


In [4]:
player_match = player_match.sort_values(
    by=["batter", "match_id"]
)


In [5]:
player_match["avg_runs_last_5"] = (
    player_match
    .groupby("batter")["runs"]
    .rolling(window=5, min_periods=1)
    .mean()
    .reset_index(level=0, drop=True)
)

player_match.head(10)


,match_id,batter,runs,balls_faced,got_out,strike_rate,avg_runs_last_5
4299,548346,A Ashish Reddy,10,10,1,100.000000,10.00
4390,548352,A Ashish Reddy,3,3,1,100.000000,6.50
4496,548359,A Ashish Reddy,8,8,1,100.000000,7.00
4699,548373,A Ashish Reddy,10,4,0,250.000000,7.75
4747,548376,A Ashish Reddy,4,5,1,80.000000,7.00
4866,598000,A Ashish Reddy,7,4,0,175.000000,6.40
4933,598004,A Ashish Reddy,14,12,1,116.666667,8.60
5027,598010,A Ashish Reddy,16,9,1,177.777778,10.20
5076,598013,A Ashish Reddy,4,5,1,80.000000,9.00
5160,598018,A Ashish Reddy,19,15,0,126.666667,12.00


In [6]:
# Take match-level context for each player-match
context = (
    df.groupby(["match_id", "batter"])
    .agg(
        batting_team=("batting_team", "first"),
        bowling_team=("bowling_team", "first"),
        inning=("inning", "first")
    )
    .reset_index()
)

context.head()


,match_id,batter,batting_team,bowling_team,inning
0,335982,AA Noffke,Royal Challengers Bangalore,Kolkata Knight Riders,2
1,335982,B Akhil,Royal Challengers Bangalore,Kolkata Knight Riders,2
2,335982,BB McCullum,Kolkata Knight Riders,Royal Challengers Bangalore,1
3,335982,CL White,Royal Challengers Bangalore,Kolkata Knight Riders,2
4,335982,DJ Hussey,Kolkata Knight Riders,Royal Challengers Bangalore,1


In [7]:
player_match = player_match.merge(
    context,
    on=["match_id", "batter"],
    how="left"
)

player_match.head()


,match_id,batter,runs,balls_faced,got_out,strike_rate,avg_runs_last_5,batting_team,bowling_team,inning
0,548346,A Ashish Reddy,10,10,1,100.0,10.00,Deccan Chargers,Mumbai Indians,1
1,548352,A Ashish Reddy,3,3,1,100.0,6.50,Deccan Chargers,Chennai Super Kings,2
2,548359,A Ashish Reddy,8,8,1,100.0,7.00,Deccan Chargers,Kings XI Punjab,2
3,548373,A Ashish Reddy,10,4,0,250.0,7.75,Deccan Chargers,Rajasthan Royals,2
4,548376,A Ashish Reddy,4,5,1,80.0,7.00,Deccan Chargers,Royal Challengers Bangalore,1


In [8]:
# Sort again to ensure correct order
player_match = player_match.sort_values(
    by=["batter", "match_id"]
)

# Target: runs in the next match
player_match["target_runs"] = (
    player_match.groupby("batter")["runs"].shift(-1)
)

player_match.head(10)


,match_id,batter,runs,balls_faced,got_out,strike_rate,avg_runs_last_5,batting_team,bowling_team,inning,target_runs
0,548346,A Ashish Reddy,10,10,1,100.000000,10.00,Deccan Chargers,Mumbai Indians,1,3.0
1,548352,A Ashish Reddy,3,3,1,100.000000,6.50,Deccan Chargers,Chennai Super Kings,2,8.0
2,548359,A Ashish Reddy,8,8,1,100.000000,7.00,Deccan Chargers,Kings XI Punjab,2,10.0
3,548373,A Ashish Reddy,10,4,0,250.000000,7.75,Deccan Chargers,Rajasthan Royals,2,4.0
4,548376,A Ashish Reddy,4,5,1,80.000000,7.00,Deccan Chargers,Royal Challengers Bangalore,1,7.0
5,598000,A Ashish Reddy,7,4,0,175.000000,6.40,Sunrisers Hyderabad,Pune Warriors,1,14.0
6,598004,A Ashish Reddy,14,12,1,116.666667,8.60,Sunrisers Hyderabad,Royal Challengers Bangalore,2,16.0
7,598010,A Ashish Reddy,16,9,1,177.777778,10.20,Sunrisers Hyderabad,Delhi Daredevils,2,4.0
8,598013,A Ashish Reddy,4,5,1,80.000000,9.00,Sunrisers Hyderabad,Kolkata Knight Riders,2,19.0
9,598018,A Ashish Reddy,19,15,0,126.666667,12.00,Sunrisers Hyderabad,Pune Warriors,1,7.0


In [9]:
player_match = player_match.dropna(subset=["target_runs"])

player_match.shape


(15842, 11)

In [10]:
player_match.to_csv("processed_player_dataset.csv", index=False)
